# Seminar HCI and BCI in practice
## Session 7 Evaluation

***In this session the performance of the SVM classifier will be evaluated.***


In [32]:
import numpy as np
import os
import pickle
from scipy import stats
from nearly import nearly
import matplotlib.pyplot as plt
from classification_svm_session07 import classification_svm
from gen_selector import gen_selector
from sklearn.svm import SVC
from sklearn.metrics import roc_curve, auc

main_path = os.getcwd()
data_path = os.path.join(main_path, 'data')
print(f'Now you are located: {main_path}')

In [33]:
ecog_file = os.path.join(data_path, 'raw/ecogStruct3.pkl')
with open(ecog_file, 'rb') as f:
    ecog = pickle.load(f)

# Take a look into our data again
print("ecog contains")
for key, value in ecog.items():
    print(f"Key:{key}, Type:{type(value)}")

# Load epoch info
epoch_file = os.path.join(data_path, 'raw/epoch2.pkl')
with open(epoch_file, 'rb') as f:
    epoch = pickle.load(f)

print("\nepoch info:")
for key, value in epoch.items():
    print(f"Key:{key}, Type:{type(value)}")

## 1. Define the dataset for the classification based on previously found features (use your results from Session 5)

In [34]:
# Frequency features (4-58, 62-118, 122-178 Hz based on Session 5 results)
# You can change the values here, based on your results from Session 5 (t-value plot)
freqBand = np.concatenate([np.arange(62, 119), np.arange(122, 179)])  # just for faster testing

# Find nearest frequency indices
freqIdx = np.unique(nearly(freqBand, ecog['periodogram']['centerFrequency']))
# Alternative:
# freqIdx = np.unique([np.argmin(np.abs(ecog['periodogram']['centerFrequency'] - f)) for f in freqBand])
nFreq = len(freqIdx)

# Number of trials with finger movement
nTrials = np.array(ecog['periodogram']['periodogram']).shape[2]

# Channel features (based on Session 5 results)
chan = np.array([17, 23, 39]) - 1  # Convert to 0-based indexing
nChan = len(chan)

# Prepare data for z-scoring (same as Session 4)
# Reshape to (nFreq, nChan*nTrials)
dat = np.array(ecog['periodogram']['periodogram'])[freqIdx, :, :][:, chan, :]
dat = dat.reshape(nFreq, nChan * nTrials, order='F')

# Z-score data along frequency axis
dat = stats.zscore(dat, axis=1) 

# Reshape data back to original structure with permutations
dat = dat.reshape(nFreq, nChan, nTrials, order='F')
dat = np.transpose(dat, (2, 1, 0)) 
dat = dat.reshape(nTrials, nFreq * nChan, order='F')

# Get the true labels for data
realClassLabels = np.array(epoch['label'])

# Take a look into the data shape now, and understand what are the dimensions of data
print("Understand the dimensions of data by yourself")
print(f"dat shape now is: {dat.shape}")
print(f"real labels for data now is: {realClassLabels.shape}")
print(f"Number of 21 class labels:{np.count_nonzero(realClassLabels == 21)}")
print(f"Number of 20 class labels:{np.count_nonzero(realClassLabels == 20)}")

---

## 2 Classification error

### 2.1 Cross Validation

In [35]:
CV_steps = 10  # CV-steps
rep_nr = 3  # performing the same classification x times (you can change this value)

svm_results_all = []

for i in range(rep_nr):
    print(f'CV-Repeat #{i+1}')
    selector = gen_selector(len(realClassLabels), CV_steps, random_seed=i)

    # SVM Model in repetition #i
    svm_results = classification_svm(dat, realClassLabels, selector, CV_steps, optimizeC=True, dispC=False)
 
    # Save SVM Results into a list, each element of the list is a dict, which contains svm results in single rep
    svm_results_all.append(svm_results)

for key, value in svm_results_all[0].items():
    print(f"Key:{key}, Type:{type(value)}")


<h2 style="color: #FF0000; font-weight: bold;">TASK 1 Discussion (2 pt):</h2>

- Explain the outputs: What information can you get from dict `svm_results`? (2 pt)

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>


In [36]:
# Plot mean and standard error of accuracy across X repetitive classifications

accuracy_svm = [result['accuracy'] for result in svm_results_all]
mean_acc = np.mean(accuracy_svm)
std_error = np.std(accuracy_svm) / np.sqrt(len(accuracy_svm))

plt.figure(figsize=(8, 6))
plt.errorbar(0, mean_acc, yerr=std_error, 
            fmt='x', color='red', 
            markersize=10, capsize=5,
            label='SVM Accuracy')

plt.xlim(-0.5, 0.5)
plt.ylim(mean_acc-2*std_error, mean_acc+2*std_error)  
plt.xticks([]) 
plt.ylabel('Classification Accuracy')
plt.title('SVM Classification Performance')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)


plt.text(0.1, mean_acc+0.2*std_error, 
         f'Mean: {mean_acc:.3f}\nSE: {std_error:.3f}',
         ha='left', va='bottom')

plt.tight_layout()
plt.show()

In [37]:
best_Cs_all = np.array([result['best_Cs'] for result in svm_results_all])
best_Cs_1d = best_Cs_all.ravel()

plt.figure(figsize=(10, 6))  
n, bins, patches = plt.hist(best_Cs_1d, 
                           bins='auto',  
                           color='skyblue',
                           edgecolor='black',
                           alpha=0.7)
plt.xlabel('C Values', fontsize=12)
plt.ylabel('Frequency', fontsize=12)
plt.title('Distribution of C Values', fontsize=14, pad=20)
plt.grid(axis='y', linestyle='--', alpha=0.7)

for i in range(len(n)):
    if n[i] > 0:  
        plt.text(bins[i] + (bins[i+1]-bins[i])/2, n[i], 
                str(int(n[i])), 
                ha='center', 
                va='bottom')

plt.tight_layout() 
plt.show()

---

### 2.2 SVM weights: FIND BEST CROSS VALIDATION

<h2 style="color: #FF0000; font-weight: bold;">TASK 2 Find best CV-Step Code (2 pt)</h2>

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration (finish the ... part in the following code cell): </h3>


In [9]:
# # Choose the fold step and repetition containing the highest accuracy
# Hint 1: `accuracy_svm_all` is a list, in which the accuracy rates over all folds and repetitions are saved
# Hint 2: check function `argmax` in numpy (https://numpy.org/devdocs/reference/generated/numpy.argmax.html 27/05/25)
# Hint 3: check function `np.unravel_index` in numpy (https://numpy.org/doc/stable/reference/generated/numpy.unravel_index.html 12/06/25)
accuracies_svm_all = [result['accuracies'] for result in svm_results_all]
max_cv_accuracy = ...
best_cv_idx = ...
best_rep_idx, best_cv_idx = np.unravel_index(best_cv_idx, (rep_nr, CV_steps))

### 2.2.2 Plot the weights for best W VECTOR:


In [ ]:
# `weights_all` saves all weights from all repetitions and all CVs
weights_all = np.array([result['weights'] for result in svm_results_all])
# Select the weight-value for the best rep and best CV
best_w = weights_all[best_rep_idx, best_cv_idx,:]

reshaped_w = best_w.reshape(nFreq, nChan, order = 'c').T

plt.figure(figsize=(12, 8))
img = plt.imshow(abs(reshaped_w), aspect='auto', cmap='RdBu_r')

freqTicks = np.arange(freqBand[0], freqBand[1] + 1, 10)
tickPos = np.linspace(0, nFreq - 1, len(freqTicks), dtype=int)

plt.yticks(ticks=np.arange(nChan), labels=chan, fontsize=12)
plt.xticks(ticks=tickPos, labels=freqTicks, fontsize=12)
plt.xlabel('Frequency (Hz)', fontsize=14, fontweight='bold')
plt.ylabel('Electrode', fontsize=14, fontweight='bold')
plt.colorbar(img)

plt.tight_layout()
plt.show()

### 2.2.3 Plot AVERAGE w vector:

In [ ]:
# Average W vector
avW = np.mean(weights_all, axis = (0,1))
reshaped_avW = avW.reshape(nFreq, nChan, order = 'c').T

plt.figure(figsize=(12, 8))
img = plt.imshow(abs(reshaped_avW), aspect='auto', cmap='RdBu_r')

freqTicks = np.arange(freqBand[0], freqBand[1] + 1, 10)
tickPos = np.linspace(0, nFreq - 1, len(freqTicks), dtype=int)

plt.yticks(ticks=np.arange(nChan), labels=chan, fontsize=12)
plt.xticks(ticks=tickPos, labels=freqTicks, fontsize=12)
plt.xlabel('Frequency (Hz)', fontsize=14, fontweight='bold')
plt.ylabel('Electrode', fontsize=14, fontweight='bold')
plt.colorbar(img)

plt.tight_layout()
plt.show()


<h2 style="color: #FF0000; font-weight: bold;">TASK 3 Discussion (1 pt)</h2>

- Explain what you see in these plots.

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>


## 2.3 Area under curve (ROC: receiver operation characteristics)

In [11]:
auc, *_ = zip(*[plot_ROC(result, realClassLabels, do_plot=True) for result in svm_results_all])

<h2 style="color: #FF0000; font-weight: bold;">TASK 4 Discussion (1 pt)</h2>

- questions in regards to plots created by the function plot_ROC above

1. What does the first plot (distances to the hyperplane) portray.
2. How would this plot ideally look like?
3. What information can you get from the second plot?
4. What is a ROC?
5. What does the Area under the ROC mean?
6. How would this plot ideally look like?

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>

## 2.4 Estimation of the chance level: Estimating Chance Level & CV

You have already tuned the `best C`, now using the `best repetition` and `best CV-step` to index the `best C-value`

In [38]:
# Now we want to bootstrap the data
# Redefine some variables, in case you have already forgot the parameters
CV_step = 10  # CV-steps
rep_nr = 400  # performing the same classification x times (now is the bootstrapping times)
realClassLabels = np.array(epoch['label'])
accuracy_permuted = np.zeros(rep_nr)
best_C = best_Cs_all[best_rep_idx, best_cv_idx]

for i in range(rep_nr):
    # print(f'CV-Repeat #{i+1}')
    # randomly generate CV-fold series for the data
    selector = gen_selector(len(realClassLabels), CV_step, random_seed=i)  #using index as random_seed for reprocduce of data results

    # SVM Model in repetition #i
    for k in range(1, CV_step + 1):
        
        # Split data into train/test sets
        testIdx = np.where(selector == k)[0]
        trainIdx = np.setdiff1d(np.arange(len(realClassLabels)), testIdx)

        X_train = dat[trainIdx, :]
        y_train = realClassLabels[trainIdx]
        X_test = dat[testIdx, :]
        y_test = realClassLabels[testIdx]

        #!!! Key step of this permutation test
        # Permute (shuffle) y_train to break the true relationship with X_train
        y_train_permuted = np.random.permutation(y_train)  # Randomly shuffles labels

        # Random label test
        svm_permuted = SVC(kernel='linear', C=best_C)
        svm_permuted.fit(X_train, y_train_permuted)  # Train on shuffled labels
    score = svm_permuted.score(X_test, y_test)
    accuracy_permuted[i] = score


In [ ]:
# plot multiple errorbars
methods = ['SVM', 'SVM permutation_test']
means = [np.mean(accuracy_svm), np.mean(accuracy_permuted)]
stds = [np.std(accuracy_svm), np.std(accuracy_permuted)]
colors = ['#1f77b4', '#ff7f0e']
text_pos = [0.12, 1.03]

plt.figure(figsize=(8, 6))
for i in range(len(methods)):
    plt.errorbar(methods[i], means[i], yerr=1.96*np.array(stds[i]),
                 fmt='x', markersize=10, capsize=5, capthick=2, color=colors[i])
plt.ylim(min(means[0]-3*stds[0], means[1]-3*stds[1]), 
         max(means[0]+3*stds[0], means[1]+3*stds[1]))
plt.xlim(-0.2, 1.65)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison (95% CI)')
for i in range(len(methods)):
    plt.text(text_pos[i], means[i]-0*stds[i], 
             f'{methods[i]}:\n {means[i]:.3f} ± {1.96*stds[i]:.3f}',
             ha='left', va='center', color=colors[i])

plt.grid(True, alpha=0.3)
plt.show()

---

<h2 style="color: #FF0000; font-weight: bold;">TASK 5 Discussion (2 pt)</h2>

- How is the chance level estimated and for what do you use it here?
- Why is this an estimation of the chance level?

<h3 style="color: #FF0000; font-weight: bold;">Your Answers or Code demostration: </h3>
